# Support Vector Regression (SVR) s RBF jádrem

## Co je SVR s RBF jádrem?

Support Vector Regression (SVR) je varianta Support Vector Machines (SVM) určená pro regresní úlohy. SVR s RBF (Radial Basis Function) jádrem využívá nelineární transformaci dat do vysoce-dimenzionálního prostoru příznaků, kde vytváří lineární regresní model.

### Princip SVR s RBF jádrem:

SVR s RBF jádrem aplikuje kernelový trik, který umožňuje efektivně pracovat ve vysoce-dimenzionálním prostoru bez explicitního výpočtu transformace. Vytváří "epsilon-trubici" kolem predikční funkce a minimalizuje chyby, které přesahují tuto trubici.

### Matematický popis:

RBF jádrová funkce je definována jako:

$$K(x, x') = \exp\left(-\gamma \|x - x'\|^2\right)$$

kde:
- $x$ a $x'$ jsou dva datové body
- $\gamma$ je parametr, který určuje vliv jednotlivých trénovacích vzorků

SVR s RBF jádrem minimalizuje objektivní funkci:

$$\min_{w, b} \frac{1}{2} \|w\|^2 + C \sum_{i=1}^{n} (\xi_i + \xi_i^*)$$

Za podmínek:
- $y_i - f(x_i) \leq \varepsilon + \xi_i$
- $f(x_i) - y_i \leq \varepsilon + \xi_i^*$
- $\xi_i, \xi_i^* \geq 0$

kde $f(x)$ je funkce s RBF jádrem: $f(x) = \sum_{i=1}^{n} (\alpha_i - \alpha_i^*) K(x_i, x) + b$

### Hlavní parametry SVR s RBF jádrem:

1. **C**: Regularizační parametr, který kontroluje trade-off mezi hladkostí predikční funkce a velikostí povolených odchylek větších než epsilon.
2. **epsilon (ε)**: Definuje šířku epsilon-trubice, v rámci které nejsou penalizovány žádné chyby.
3. **gamma (γ)**: Určuje vliv jednotlivých trénovacích vzorků; vyšší hodnoty gamma vedou ke "užšímu" vlivu každého vzorku.

### Kdy použít SVR s RBF jádrem:

- Když jsou vztahy mezi příznaky a cílovou proměnnou komplexní a nelineární
- Když potřebujete vysokou přesnost predikce
- Když máte rozumně velkou datovou sadu (stovky až tisíce vzorků)
- Když potřebujete model odolný vůči odlehlým hodnotám

In [ ]:
# Import potřebných knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.datasets import make_regression, fetch_california_housing, load_boston
from sklearn.linear_model import LinearRegression, Ridge
import time

# Nastavení pro reprodukovatelnost
np.random.seed(42)

# Nastavení vizualizací
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_palette('viridis')

## 1. Jednoduché vysvětlení SVR s RBF jádrem na syntetických datech

Nejprve si vytvoříme jednoduchý syntetický dataset, abychom ilustrovali, jak SVR s RBF jádrem funguje a jak se jeho chování mění v závislosti na parametrech.

In [ ]:
# Vytvoření jednoduchého syntetického datasetu s nelineárním vztahem
def create_nonlinear_data(n_samples=100, noise=0.4, random_state=42):
    np.random.seed(random_state)
    X = np.sort(np.random.rand(n_samples) * 6 - 3)
    y = np.sin(X) + 0.5 * X + np.random.randn(n_samples) * noise
    X = X.reshape(-1, 1)  # Převod na 2D array pro sklearn
    return X, y

# Vytvoření dat
X_simple, y_simple = create_nonlinear_data(n_samples=100, noise=0.4)

# Vizualizace dat
plt.figure(figsize=(10, 6))
plt.scatter(X_simple, y_simple, alpha=0.7)
plt.title('Syntetická data s nelineárním vztahem')
plt.xlabel('X')
plt.ylabel('y')
plt.grid(True)
plt.show()

In [ ]:
# Rozdělení na trénovací a testovací data
X_train, X_test, y_train, y_test = train_test_split(X_simple, y_simple, test_size=0.2, random_state=42)

# Vytvoření SVR modelu s RBF jádrem s výchozími parametry
svr_rbf = SVR(kernel='rbf')
svr_rbf.fit(X_train, y_train)

# Vytvoření dat pro vizualizaci predikce
X_plot = np.linspace(-3, 3, 100).reshape(-1, 1)
y_pred_rbf = svr_rbf.predict(X_plot)

# Vyhodnocení na testovacích datech
y_pred_test = svr_rbf.predict(X_test)
mse_rbf = mean_squared_error(y_test, y_pred_test)
r2_rbf = r2_score(y_test, y_pred_test)

print(f"SVR s RBF jádrem (výchozí parametry):")
print(f"  MSE: {mse_rbf:.4f}")
print(f"  R²: {r2_rbf:.4f}")

# Vizualizace výsledků
plt.figure(figsize=(12, 6))
plt.scatter(X_train, y_train, alpha=0.7, label='Trénovací data')
plt.scatter(X_test, y_test, alpha=0.7, marker='x', color='red', label='Testovací data')
plt.plot(X_plot, y_pred_rbf, color='blue', linewidth=2, label='SVR s RBF jádrem')

# Porovnání s lineárním SVR
svr_linear = SVR(kernel='linear')
svr_linear.fit(X_train, y_train)
y_pred_linear = svr_linear.predict(X_plot)
plt.plot(X_plot, y_pred_linear, color='green', linestyle='--', linewidth=2, label='SVR s lineárním jádrem')

plt.title('SVR s RBF jádrem vs. SVR s lineárním jádrem')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

### Vliv parametrů C, epsilon a gamma na SVR s RBF jádrem

Prozkoumáme, jak tři klíčové parametry SVR s RBF jádrem ovlivňují výsledný model:
1. **C** - regularizační parametr
2. **gamma** - parametr RBF jádra
3. **epsilon** - šířka "trubice" tolerance chyb

In [ ]:
# Analýza vlivu parametrů C a gamma
C_values = [0.1, 1.0, 10.0, 100.0]
gamma_values = [0.01, 0.1, 1.0, 10.0]

# Vytvoření mřížky grafů
fig, axes = plt.subplots(len(C_values), len(gamma_values), figsize=(16, 12))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

for i, C in enumerate(C_values):
    for j, gamma in enumerate(gamma_values):
        ax = axes[i, j]
        
        # Trénování SVR s konkrétními parametry
        svr = SVR(kernel='rbf', C=C, gamma=gamma, epsilon=0.1)
        svr.fit(X_train, y_train)
        
        # Predikce
        y_pred = svr.predict(X_plot)
        
        # Vyhodnocení
        y_pred_test_param = svr.predict(X_test)
        mse = mean_squared_error(y_test, y_pred_test_param)
        
        # Vizualizace
        ax.scatter(X_train[:, 0], y_train, alpha=0.4, s=30)
        ax.plot(X_plot, y_pred, color='red', linewidth=2)
        
        ax.set_title(f"C={C}, γ={gamma}, MSE={mse:.4f}")
        ax.grid(True)
        
        # Nastavení stejných os pro lepší srovnání
        ax.set_xlim(-3, 3)
        ax.set_ylim(min(y_simple) - 0.5, max(y_simple) + 0.5)
        
        # Popisky os pouze na okrajových grafech
        if i == len(C_values) - 1:
            ax.set_xlabel('X')
        if j == 0:
            ax.set_ylabel('y')

plt.suptitle('Vliv parametrů C a gamma na SVR s RBF jádrem', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
# Analýza vlivu parametru epsilon
epsilon_values = [0.01, 0.1, 0.5, 1.0, 2.0]

# Vytvoření grafu
plt.figure(figsize=(16, 10))

for i, epsilon in enumerate(epsilon_values):
    # Trénování SVR s různými hodnotami epsilon
    svr = SVR(kernel='rbf', C=1.0, gamma=0.1, epsilon=epsilon)
    svr.fit(X_train, y_train)
    
    # Predikce
    y_pred = svr.predict(X_plot)
    
    # Vyhodnocení
    y_pred_test_param = svr.predict(X_test)
    mse = mean_squared_error(y_test, y_pred_test_param)
    
    # Vytvoření subplotu
    plt.subplot(2, 3, i + 1)
    plt.scatter(X_train[:, 0], y_train, alpha=0.4, s=30)
    plt.plot(X_plot, y_pred, color='red', linewidth=2)
    
    # Vykreslení epsilon-trubice
    plt.fill_between(X_plot.ravel(),
                     y_pred - epsilon,
                     y_pred + epsilon,
                     alpha=0.2, color='gray')
    
    plt.title(f"epsilon={epsilon}, MSE={mse:.4f}")
    plt.grid(True)
    plt.xlim(-3, 3)
    plt.ylim(min(y_simple) - 0.5, max(y_simple) + 0.5)

plt.suptitle('Vliv parametru epsilon na SVR s RBF jádrem', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 2. Porovnání SVR s RBF jádrem s jinými regresními modely

Nyní porovnáme SVR s RBF jádrem s dalšími regresními modely na složitějším syntetickém datasetu s více příznaky.

In [ ]:
# Vytvoření složitějšího datasetu s více příznaky a nelineárními vztahy
def create_complex_dataset(n_samples=200, n_features=2, random_state=42):
    rng = np.random.RandomState(random_state)
    X = rng.rand(n_samples, n_features) * 4 - 2
    y = (np.sin(X[:, 0]) + 0.5 * X[:, 0] + 
         np.where(n_features >= 2, np.cos(X[:, 1]) + 0.5 * X[:, 1], 0) +
         rng.randn(n_samples) * 0.2)
    return X, y

# Vytvoření datasetu
X_complex, y_complex = create_complex_dataset(n_samples=300, n_features=3)

# Rozdělení na trénovací a testovací data
X_train_complex, X_test_complex, y_train_complex, y_test_complex = train_test_split(
    X_complex, y_complex, test_size=0.2, random_state=42)

# Standardizace dat
scaler = StandardScaler()
X_train_complex_scaled = scaler.fit_transform(X_train_complex)
X_test_complex_scaled = scaler.transform(X_test_complex)

# Definice modelů pro porovnání
models = {
    'Lineární regrese': LinearRegression(),
    'Ridge regrese': Ridge(alpha=1.0),
    'SVR (lineární)': SVR(kernel='linear', C=1.0),
    'SVR (RBF)': SVR(kernel='rbf', C=10.0, gamma=0.1),
    'SVR (polynomiální)': SVR(kernel='poly', C=10.0, degree=3)
}

# Trénování a vyhodnocení modelů
results = []

for name, model in models.items():
    # Měření času trénování
    start_time = time.time()
    model.fit(X_train_complex_scaled, y_train_complex)
    train_time = time.time() - start_time
    
    # Predikce
    y_pred = model.predict(X_test_complex_scaled)
    
    # Výpočet metrik
    mse = mean_squared_error(y_test_complex, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_complex, y_pred)
    r2 = r2_score(y_test_complex, y_pred)
    
    # Uložení výsledků
    results.append({
        'Model': name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2,
        'Čas trénování (s)': train_time
    })

# Převod výsledků na DataFrame a výpis
results_df = pd.DataFrame(results)
print("Srovnání regresních modelů:")
print(results_df.round(4))

In [ ]:
# Vizualizace výsledků srovnání
plt.figure(figsize=(14, 8))

# Vizualizace metrik
metrics = ['MSE', 'RMSE', 'MAE']
plt.subplot(1, 2, 1)

for i, metric in enumerate(metrics):
    plt.bar([x + i*0.2 for x in range(len(models))], results_df[metric], width=0.2, 
            label=metric, alpha=0.7)
    
plt.xticks([x + 0.2 for x in range(len(models))], results_df['Model'], rotation=45)
plt.ylabel('Hodnota metriky')
plt.title('Srovnání metrik chyb')
plt.legend()
plt.grid(True, axis='y')

# Vizualizace R2
plt.subplot(1, 2, 2)
bars = plt.bar(results_df['Model'], results_df['R²'], color='green', alpha=0.7)
plt.ylabel('R² skóre')
plt.title('Srovnání R² skóre')
plt.grid(True, axis='y')
plt.xticks(rotation=45)

# Přidání hodnot na vrcholy sloupců
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{height:.3f}', ha='center', va='bottom')
    
plt.tight_layout()
plt.show()

## 3. Optimalizace hyperparametrů SVR s RBF jádrem

Pro dosažení optimálního výkonu SVR s RBF jádrem je klíčové správně nastavit hyperparametry C, gamma a epsilon.

In [ ]:
# Definice parametrů pro GridSearchCV
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.01, 0.1, 1, 'scale', 'auto'],
    'epsilon': [0.01, 0.1, 0.5]
}

# Vytvoření pipeline s předzpracováním a SVR
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf'))
])

# Nastavení GridSearchCV
param_grid_pipeline = {
    'svr__C': param_grid['C'],
    'svr__gamma': param_grid['gamma'],
    'svr__epsilon': param_grid['epsilon']
}

# Pro rychlejší demonstraci použijeme menší část syntetických dat
X_sample, y_sample = create_complex_dataset(n_samples=200, n_features=2)
X_train_sample, X_test_sample, y_train_sample, y_test_sample = train_test_split(
    X_sample, y_sample, test_size=0.2, random_state=42)

# Nastavení GridSearchCV
grid_search = GridSearchCV(
    pipeline, param_grid_pipeline, cv=3, 
    scoring='neg_mean_squared_error', verbose=1, n_jobs=-1
)

# Spuštění Grid Search
grid_search.fit(X_train_sample, y_train_sample)

# Výpis nejlepších parametrů
print(f"Nejlepší parametry: {grid_search.best_params_}")
print(f"Nejlepší skóre (neg MSE): {grid_search.best_score_:.4f}")

# Evaluace nejlepšího modelu
best_pipeline = grid_search.best_estimator_
y_pred_best = best_pipeline.predict(X_test_sample)

mse_best = mean_squared_error(y_test_sample, y_pred_best)
r2_best = r2_score(y_test_sample, y_pred_best)

print(f"\nVýkon nejlepšího SVR modelu na testovacích datech:")
print(f"  MSE: {mse_best:.4f}")
print(f"  RMSE: {np.sqrt(mse_best):.4f}")
print(f"  R²: {r2_best:.4f}")

## 4. Aplikace SVR s RBF jádrem na reálný dataset

Nyní aplikujeme SVR s RBF jádrem na dataset California Housing.

In [ ]:
# Načtení California Housing datasetu
housing = fetch_california_housing()
X_housing = housing.data
y_housing = housing.target
feature_names = housing.feature_names

# Informace o datasetu
print(f"California Housing Dataset:")
print(f"  Počet vzorků: {X_housing.shape[0]}")
print(f"  Počet příznaků: {X_housing.shape[1]}")
print(f"  Příznaky: {feature_names}")
print("\nPopisná statistika cílové proměnné (mediánová cena domu):")
print(f"  Minimum: {y_housing.min():.2f}")
print(f"  Maximum: {y_housing.max():.2f}")
print(f"  Průměr: {y_housing.mean():.2f}")
print(f"  Medián: {np.median(y_housing):.2f}")
print(f"  Směrodatná odchylka: {y_housing.std():.2f}")

# Pro urychlení výpočtů použijeme podmnožinu dat
# V reálné aplikaci byste mohli využít celý dataset
np.random.seed(42)
indices = np.random.choice(X_housing.shape[0], 3000, replace=False)
X_housing_subset = X_housing[indices]
y_housing_subset = y_housing[indices]

# Rozdělení na trénovací a testovací data
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_housing_subset, y_housing_subset, test_size=0.2, random_state=42)

In [ ]:
# Vytvoření pipeline s předzpracováním a SVR s RBF jádrem
# Použijeme parametry, které jsme našli dříve (nebo je můžete přizpůsobit)
rbf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=10.0, gamma=0.1, epsilon=0.1))
])

# Trénování modelu
start_time = time.time()
rbf_pipeline.fit(X_train_h, y_train_h)
train_time = time.time() - start_time

# Predikce
y_pred_h = rbf_pipeline.predict(X_test_h)

# Vyhodnocení výkonu
mse_h = mean_squared_error(y_test_h, y_pred_h)
rmse_h = np.sqrt(mse_h)
mae_h = mean_absolute_error(y_test_h, y_pred_h)
r2_h = r2_score(y_test_h, y_pred_h)

print(f"SVR s RBF jádrem na California Housing datasetu:")
print(f"  Čas trénování: {train_time:.2f} sekund")
print(f"  MSE: {mse_h:.4f}")
print(f"  RMSE: {rmse_h:.4f}")
print(f"  MAE: {mae_h:.4f}")
print(f"  R²: {r2_h:.4f}")

In [ ]:
# Porovnání s jinými modely na California Housing datasetu
housing_models = {
    'Lineární regrese': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Ridge regrese': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'SVR (lineární)': Pipeline([('scaler', StandardScaler()), ('model', SVR(kernel='linear', C=1.0))]),
    'SVR (RBF)': Pipeline([('scaler', StandardScaler()), ('model', SVR(kernel='rbf', C=10.0, gamma=0.1))]),
    'SVR (polynomiální)': Pipeline([('scaler', StandardScaler()), ('model', SVR(kernel='poly', degree=2, C=10.0))])
}

housing_results = []

for name, model in housing_models.items():
    # Trénování
    start_time = time.time()
    model.fit(X_train_h, y_train_h)
    train_time = time.time() - start_time
    
    # Predikce
    y_pred = model.predict(X_test_h)
    
    # Metriky
    mse = mean_squared_error(y_test_h, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_h, y_pred)
    r2 = r2_score(y_test_h, y_pred)
    
    housing_results.append({
        'Model': name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2,
        'Čas trénování (s)': train_time
    })

housing_results_df = pd.DataFrame(housing_results)
print("Porovnání modelů na California Housing datasetu:")
print(housing_results_df.round(4).sort_values('MSE'))

In [ ]:
# Vizualizace porovnání modelů na California Housing datasetu
plt.figure(figsize=(14, 10))

# Seřazení modelů podle výkonu
sorted_results = housing_results_df.sort_values('MSE')

# Vizualizace MSE
plt.subplot(2, 2, 1)
sns.barplot(x='Model', y='MSE', data=sorted_results, palette='viridis')
plt.title('MSE pro různé modely')
plt.xticks(rotation=45)
plt.grid(axis='y')

# Vizualizace R²
plt.subplot(2, 2, 2)
sns.barplot(x='Model', y='R²', data=sorted_results.sort_values('R²', ascending=False), palette='viridis')
plt.title('R² pro různé modely')
plt.xticks(rotation=45)
plt.grid(axis='y')

# Vizualizace času trénování
plt.subplot(2, 2, 3)
sns.barplot(x='Model', y='Čas trénování (s)', data=sorted_results, palette='viridis')
plt.title('Čas trénování modelů')
plt.xticks(rotation=45)
plt.grid(axis='y')

# Vizualizace skutečných vs. predikovaných hodnot pro SVR (RBF)
plt.subplot(2, 2, 4)
svr_rbf_model = housing_models['SVR (RBF)']
y_pred_rbf = svr_rbf_model.predict(X_test_h)
plt.scatter(y_test_h, y_pred_rbf, alpha=0.5)
plt.plot([y_housing_subset.min(), y_housing_subset.max()], 
         [y_housing_subset.min(), y_housing_subset.max()], 'r--')
plt.xlabel('Skutečné hodnoty')
plt.ylabel('Predikované hodnoty')
plt.title('SVR (RBF): Skutečné vs. Predikované hodnoty')
plt.grid(True)

plt.tight_layout()
plt.show()

## 5. Podrobné zkoumání vlivu šumu a outlierů na SVR s RBF jádrem

Jednou z hlavních výhod SVR je jeho robustnost vůči odlehlým hodnotám (outlierům). Pojďme zkoumat, jak se SVR s RBF jádrem vyrovnává s různými úrovněmi šumu a outlierů v datech.

In [ ]:
# Vytvoření datasetu s různými úrovněmi šumu a outlierů
def create_dataset_with_outliers(n_samples=100, noise=0.2, outlier_ratio=0.1, outlier_strength=2.0, random_state=42):
    rng = np.random.RandomState(random_state)
    # Základní data
    X = np.sort(rng.rand(n_samples) * 4 - 2)
    y = np.sin(X) + X/2 + rng.randn(n_samples) * noise
    
    # Přidání outlierů
    n_outliers = int(n_samples * outlier_ratio)
    outlier_indices = rng.choice(range(n_samples), n_outliers, replace=False)
    y[outlier_indices] += rng.randn(n_outliers) * outlier_strength
    
    X = X.reshape(-1, 1)
    return X, y

# Vytvoření několika datasetů s různým množstvím šumu a outlierů
datasets = {
    'Základní (nízký šum)': create_dataset_with_outliers(noise=0.1, outlier_ratio=0),
    'Střední šum': create_dataset_with_outliers(noise=0.5, outlier_ratio=0),
    'Vysoký šum': create_dataset_with_outliers(noise=1.0, outlier_ratio=0),
    'S 5% outlierů': create_dataset_with_outliers(noise=0.2, outlier_ratio=0.05, outlier_strength=3.0),
    'S 15% outlierů': create_dataset_with_outliers(noise=0.2, outlier_ratio=0.15, outlier_strength=3.0)
}

# Vizualizace datasetů
plt.figure(figsize=(15, 10))
for i, (name, (X, y)) in enumerate(datasets.items()):
    plt.subplot(2, 3, i+1)
    plt.scatter(X, y, alpha=0.7)
    plt.title(name)
    plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Porovnání SVR s RBF jádrem s jinými regresními modely na datech s různým šumem a outliery
compare_models = {
    'Lineární regrese': LinearRegression(),
    'SVR (lineární)': SVR(kernel='linear', C=1.0, epsilon=0.1),
    'SVR (RBF)': SVR(kernel='rbf', C=10.0, gamma=0.1, epsilon=0.1)
}

# Výsledky pro každý dataset a model
noise_outlier_results = []

for dataset_name, (X, y) in datasets.items():
    # Rozdělení na trénovací a testovací sadu
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Standardizace dat
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Testování každého modelu
    for model_name, model in compare_models.items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        
        # Výpočet metrik
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        noise_outlier_results.append({
            'Dataset': dataset_name,
            'Model': model_name,
            'MSE': mse,
            'R²': r2
        })

# Převod na DataFrame a výpis
noise_results_df = pd.DataFrame(noise_outlier_results)
print("Výsledky modelů na datasetech s různým šumem a outliery:")

# Pivotní tabulka pro lepší porovnání
pivot_mse = noise_results_df.pivot_table(index='Dataset', columns='Model', values='MSE')
pivot_r2 = noise_results_df.pivot_table(index='Dataset', columns='Model', values='R²')

print("\nMSE pro různé modely a datasety:")
print(pivot_mse.round(4))

print("\nR² pro různé modely a datasety:")
print(pivot_r2.round(4))

In [ ]:
# Vizualizace výsledků pro jednotlivé datasety a modely
plt.figure(figsize=(18, 12))

for i, (dataset_name, (X, y)) in enumerate(datasets.items()):
    # Vytvoření dat pro predikci
    X_plot = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
    
    # Rozdělení dat a standardizace
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_plot_scaled = scaler.transform(X_plot)
    
    plt.subplot(2, 3, i+1)
    plt.scatter(X_train, y_train, alpha=0.6, label='Trénovací data')
    plt.scatter(X_test, y_test, alpha=0.6, marker='x', color='red', label='Testovací data')
    
    # Predikce jednotlivých modelů
    for model_name, model in compare_models.items():
        model.fit(X_train_scaled, y_train)
        y_plot_pred = model.predict(X_plot_scaled)
        
        # Filtrace výsledků pro tento dataset a model
        mse = noise_results_df[(noise_results_df['Dataset'] == dataset_name) & 
                              (noise_results_df['Model'] == model_name)]['MSE'].values[0]
        
        plt.plot(X_plot, y_plot_pred, label=f'{model_name}, MSE={mse:.3f}')
    
    plt.title(dataset_name)
    plt.grid(True)
    plt.legend()
    
plt.tight_layout()
plt.show()

## 6. Shrnutí a doporučení pro práci s SVR s RBF jádrem

Na základě provedených experimentů a analýz můžeme formulovat následující doporučení a shrnutí pro efektivní práci s SVR s RBF jádrem.

### Klíčové vlastnosti SVR s RBF jádrem:

1. **Schopnost modelovat nelineární vztahy** - RBF jádro umožňuje zachytit komplexní nelineární vztahy v datech díky mapování do vysoce-dimenzionálního prostoru příznaků.

2. **Odolnost vůči odlehlým hodnotám** - Díky epsilon-insensitive loss function je SVR méně citlivý na odlehlé hodnoty, což jsme viděli v experimentech s daty obsahujícími outliery.

3. **Dobrý výkon na středně velkých datových sadách** - SVR s RBF jádrem poskytuje vysokou přesnost, zejména pro datové sady střední velikosti (stovky až tisíce vzorků).

4. **Vysoká citlivost na volbu hyperparametrů** - Jak ukázaly naše experimenty, výkon SVR s RBF jádrem silně závisí na správném nastavení parametrů C, gamma a epsilon.

5. **Potřeba standardizace dat** - Pro správnou funkci SVR s RBF jádrem je téměř vždy nezbytné standardizovat vstupní data.

### Doporučení pro práci s SVR s RBF jádrem:

1. **Standardizace dat** - Vždy standardizujte příznaky před použitím SVR s RBF jádrem, ideálně pomocí `StandardScaler` v rámci pipeline.

2. **Optimalizace hyperparametrů** - Věnujte čas ladění zejména těchto parametrů:
   - **C**: Zkuste hodnoty v rozsahu [0.1, 1, 10, 100]
   - **gamma**: Zkuste hodnoty v rozsahu [0.01, 0.1, 1, 'scale', 'auto']
   - **epsilon**: Zkuste hodnoty v rozsahu [0.01, 0.1, 0.5]

3. **Vliv parametrů** - Mějte na paměti vliv jednotlivých parametrů:
   - Vyšší C = menší regularizace = složitější model s rizikem přeučení
   - Vyšší gamma = užší vliv jednotlivých vzorků = složitější model
   - Vyšší epsilon = širší "trubice" = méně podpůrných vektorů = jednodušší model

4. **Škálovatelnost** - SVR s RBF jádrem není vhodný pro velmi velké datové sady (desítky tisíc vzorků a více) kvůli výpočetní náročnosti. V takových případech zvažte:
   - Použití podvzorkování dat
   - Alternativní modely jako Gradient Boosting nebo Random Forest
   - Použití SGDRegressor s RBF aproximací

5. **Interpretovatelnost** - SVR s RBF jádrem má nízkou interpretovatelnost; pokud je interpretace modelu důležitá, zvažte SVR s lineárním jádrem nebo jiné interpretovatelné modely.

### Typické případy použití pro SVR s RBF jádrem:

1. **Nelineární regresní úlohy** - Když vztahy v datech jsou pravděpodobně nelineární.

2. **Datasety střední velikosti** - Od stovek do několika tisíc vzorků.

3. **Data s odlehlými hodnotami** - Když potřebujete robustnost vůči outlierům.

4. **Když je vysoká přesnost důležitější než interpretovatelnost** - Když je prioritou přesnost předpovědi.

5. **Když máte k dispozici výpočetní zdroje pro ladění hyperparametrů** - Protože optimalizace parametrů je klíčová pro dobré výsledky.

### Shrnutí:

SVR s RBF jádrem je výkonný regresní algoritmus pro nelineární vztahy v datech. Jeho hlavní silou je schopnost modelovat komplexní vztahy a odolnost vůči outlierům. Správné nastavení hyperparametrů a předzpracování dat jsou klíčové pro dosažení optimálních výsledků. Pro úspěšnou aplikaci SVR s RBF jádrem doporučujeme použít GridSearchCV pro nalezení optimálních parametrů a vždy standardizovat vstupní data pomocí StandardScaler.